In [1]:
library(poppr)
library(vcfR)
library(adegenet)
library(parallel)

Loading required package: adegenet

Loading required package: ade4


   /// adegenet 2.1.11 is loaded ////////////

   > overview: '?adegenet'
   > tutorials/doc/questions: 'adegenetWeb()' 
   > bug reports/feature requests: adegenetIssues()



Registered S3 method overwritten by 'pegas':
  method      from
  print.amova ade4

This is poppr version 2.9.8. To get started, type package?poppr
OMP parallel support: available


   *****       ***   vcfR   ***       *****
   This is vcfR 1.16.0 
     browseVignettes('vcfR') # Documentation
     citation('vcfR') # Citation
   *****       *****      *****       *****




In [10]:
metafile <- '../../../../hybrid_diversity/5_snp_analyses/6_hybrid_analyses/metadata.txt'
vcffile <- '../../../../hybrid_diversity/5_snp_analyses/2_atype_network/atype_variants_qualfiltered_maffiltered_renamed_justhybrids.recode.vcf'

In [11]:
cat("Loading VCF file...\n")
# Read the VCF - adjust path as needed
vcf <- read.vcfR(vcffile)
 
# Convert to genind object (required for AMOVA)
cat("Converting to genind object...\n")
gen <- vcfR2genind(vcf)
 
cat("Sample names in VCF:\n")
print(indNames(gen))

Loading VCF file...
Scanning file to determine attributes.
File attributes:
  meta lines: 65
  header_line: 66
  variant count: 347925
  column count: 31
Meta line 65 read in.
All meta lines processed.
gt matrix initialized.
Character matrix gt created.
  Character matrix gt rows: 347925
  Character matrix gt cols: 31
  skip: 0
  nrows: 347925
  row_num: 0
Processed variant: 347925
All variants processed
Converting to genind object...
Sample names in VCF:
 [1] "12A1"         "12A1-SC1"     "12A1-SC2"     "12A1-SC3"     "13A5"        
 [6] "8020"         "8027"         "ARC66"        "ARC81"        "ARC82"       
[11] "ARC83"        "ARC84"        "PK2297D4"     "PK2298C6"     "PK2299B3"    
[16] "PK2300D3"     "UTEX2797"     "UTEX2797-SC1" "UTEX2797-SC2" "UTEX2797-SC3"
[21] "UTEX2797-SC4" "UTEX2827"    


In [12]:
cat("\nLoading metadata...\n")
metadata <- read.table(metafile, sep = "\t", stringsAsFactors = FALSE, header = TRUE)
print(head(metadata))


Loading metadata...
  strain_ID cytotype    region
1      8020       A1     Texas
2     ARC66       A1 Carolinas
3     ARC81       A1 Carolinas
4     ARC82       A1 Carolinas
5     ARC83       A1 Carolinas
6     ARC84       A1 Carolinas


In [13]:
# Check that all VCF samples have metadata
missing_meta <- setdiff(indNames(gen), metadata$strain_ID)
if (length(missing_meta) > 0) {
  stop("Missing metadata for strains: ", paste(missing_meta, collapse = ", "))
}

In [14]:
# Reorder metadata to match VCF sample order
metadata <- metadata[match(indNames(gen), metadata$strain_ID), ]

In [15]:
print(head(metadata))

   strain_ID cytotype region
14      12A1       A2  Texas
15  12A1-SC1       A2  Texas
16  12A1-SC2       A2  Texas
17  12A1-SC3       A2  Texas
18      13A5       A2  Texas
1       8020       A1  Texas


In [ ]:
# Set population structure and run AMOVA without within-individual variance
setPop(gen) <- ~region/cytotype
amova_result1 <- poppr.amova(gen, ~region, method = "pegas", within = FALSE, nperm = 9, parallel = TRUE, ncpus = 4)
print(amova_result1)

Warning message:
“Cannot set the population from an empty strata”
Warning message:
“Cannot set the population from an empty strata”

 No missing values detected.




In [ ]:
# Attach metadata as strata to the genind object
strata_df <- data.frame(
  strain_ID = metadata$strain_ID,
  cytotype = factor(metadata$cytotype),
  region = factor(metadata$region)
)
strata(gen) <- strata_df
 
cat("\nStrata summary:\n")
print(table(strata_df$cytotype, strata_df$region))


In [ ]:
# Set population structure and run AMOVA without within-individual variance
setPop(gen) <- ~region/cytotype
amova_result1 <- poppr.amova(gen, ~region/cytotype, method = "pegas", within = FALSE, nperm = 9)
print(amova_result1)

In [ ]:
# Test the other hierarchy
setPop(gen) <- ~cytotype/region  
amova_result2 <- poppr.amova(gen, ~cytotype/region, method = "pegas", within = FALSE, nperm = 9)
print(amova_result2)

In [ ]:
# Check the structure of the AMOVA object
cat("\nStructure of amova_result1:\n")
str(amova_result1)

cat("\nNames of components:\n")
names(amova_result1)

In [ ]:
# The variance components are usually in:
cat("\nVariance components (Geography > Cytotype):\n")
print(amova_result1$varcomp)  # or try amova_result1$componentsofcovariance

cat("\nVariance components (Cytotype > Geography):\n")
print(amova_result2$varcomp)  # or try amova_result2$componentsofcovariance